<a href="https://colab.research.google.com/github/Harshitha82/UE25CS645BC2_PES1PG25CS082_Fashion_MNIST_CNN/blob/main/UE25CS645BC2_PES1PG25CS082.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

In [ ]:
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0
print("Train Shape:", train_images.shape)
print("Test Shape:", test_images.shape)

Train Shape: (60000, 28, 28)
Test Shape: (10000, 28, 28)


In [ ]:
class ConvLayer:

    def __init__(self, num_filters, filter_size):

        self.num_filters = num_filters
        self.filter_size = filter_size

        self.filters = np.random.randn(
            num_filters,
            filter_size,
            filter_size
        ) / (filter_size * filter_size)

    def forward(self, input):

        self.input = input

        h, w = input.shape

        output = np.zeros((
            self.num_filters,
            h - self.filter_size + 1,
            w - self.filter_size + 1
        ))

        for f in range(self.num_filters):

            for i in range(h - self.filter_size + 1):

                for j in range(w - self.filter_size + 1):

                    region = input[
                        i:i+self.filter_size,
                        j:j+self.filter_size
                    ]

                    output[f, i, j] = np.sum(
                        region * self.filters[f]
                    )

        return output

    def backward(self, d_out, learning_rate):

        d_filters = np.zeros(self.filters.shape)

        for f in range(self.num_filters):

            for i in range(d_out.shape[1]):

                for j in range(d_out.shape[2]):

                    region = self.input[
                        i:i+self.filter_size,
                        j:j+self.filter_size
                    ]

                    d_filters[f] += d_out[f, i, j] * region
        self.filters -= learning_rate * d_filters


In [ ]:
class MaxPool:

    def __init__(self, size):

        self.size = size

    def forward(self, input):

        self.input = input

        num_filters, h, w = input.shape

        output = np.zeros((
            num_filters,
            h // 2,
            w // 2
        ))

        for f in range(num_filters):

            for i in range(h // 2):

                for j in range(w // 2):

                    region = input[
                        f,
                        i*2:i*2+2,
                        j*2:j*2+2
                    ]

                    output[f, i, j] = np.max(region)

        return output

In [ ]:
def flatten(input):

    return input.flatten()

In [ ]:
class Dense:

    def __init__(self, input_len, output_len):

        self.weights = np.random.randn(
            input_len,
            output_len
        ) / input_len

        self.bias = np.zeros(output_len)

    def forward(self, input):

        self.input = input

        return np.dot(input, self.weights) + self.bias

    def backward(self, d_out, learning_rate):

        d_weights = np.outer(self.input, d_out)

        d_input = np.dot(self.weights, d_out)

        self.weights -= learning_rate * d_weights

        self.bias -= learning_rate * d_out

        return d_input

In [ ]:
def softmax(x):

    exp = np.exp(x - np.max(x))

    return exp / np.sum(exp)

In [ ]:
def cross_entropy(probs, label):

    return -np.log(probs[label])


In [ ]:
conv = ConvLayer(num_filters=8, filter_size=3)

dense = Dense(26 * 26 * 8, 10)

In [ ]:
def forward(image, label):

    out = conv.forward(image)

    out = flatten(out)

    out = dense.forward(out)

    probs = softmax(out)

    loss = cross_entropy(probs, label)

    return probs, loss

In [ ]:
def train(image, label, learning_rate=0.005):

    # forward pass
    probs, loss = forward(image, label)

    # gradient
    gradient = probs.copy()

    gradient[label] -= 1

    # dense backward
    grad_back = dense.backward(
        gradient,
        learning_rate
    )

    # reshape
    grad_back = grad_back.reshape(8, 26, 26)

    # conv backward
    conv.backward(
        grad_back,
        learning_rate
    )

    return loss

In [ ]:
print("\nTraining Started...\n")

for epoch in range(3):

    print("Epoch:", epoch + 1)

    total_loss = 0

    # training on first 1000 images
    for i in range(10000):

        image = train_images[i]

        label = train_labels[i]

        loss = train(image, label)

        total_loss += loss

        if i % 1000 == 0:

            print(
                "Step:",
                i,
                "Loss:",
                round(loss, 3)
            )

    print("Average Loss:", total_loss / 1000)

    print()


Training Started...

Epoch: 1
Step: 0 Loss: 0.0
Step: 1000 Loss: 0.069
Step: 2000 Loss: 0.079
Step: 3000 Loss: 0.016
Step: 4000 Loss: 0.041
Step: 5000 Loss: 0.118
Step: 6000 Loss: 0.042
Step: 7000 Loss: 0.003
Step: 8000 Loss: 0.371
Step: 9000 Loss: 0.024
Average Loss: 4.9107168884240595

Epoch: 2
Step: 0 Loss: 0.005
Step: 1000 Loss: 0.091
Step: 2000 Loss: 0.073
Step: 3000 Loss: 0.007
Step: 4000 Loss: 0.05
Step: 5000 Loss: 0.106
Step: 6000 Loss: 0.052
Step: 7000 Loss: 0.001
Step: 8000 Loss: 0.398
Step: 9000 Loss: 0.023
Average Loss: 4.4458706811309305

Epoch: 3
Step: 0 Loss: 0.003
Step: 1000 Loss: 0.121
Step: 2000 Loss: 0.073
Step: 3000 Loss: 0.005
Step: 4000 Loss: 0.033
Step: 5000 Loss: 0.107
Step: 6000 Loss: 0.047
Step: 7000 Loss: 0.001
Step: 8000 Loss: 0.411
Step: 9000 Loss: 0.021
Average Loss: 4.157589360526773



In [63]:
correct = 0

for i in range(1000):

    image = test_images[i]

    label = test_labels[i]

    probs, loss = forward(image, label)

    prediction = np.argmax(probs)

    if prediction == label:

        correct += 1

accuracy = (correct / 1000) * 100

print("\nFinal Accuracy: %.2f%%" % accuracy)


Final Accuracy: 81.60%
